In [ ]:
import geopandas as gpd
import igraph as ig
import numpy as np
import osmnx as ox
import pandas as pd
from geopy.distance import geodesic

from paths import paths, prepare_preprocessing
from preprocessing.src.constants import LSOA_IDS
from preprocessing.src.geo import get_street_network
from preprocessing.src.isochrone import extract_zones_by_duration
from preprocessing.src.threading import parallelize

In [ ]:
ox.settings.cache_folder = paths.preprocessing.cache

In [ ]:
prepare_preprocessing()

In [ ]:
# Reuse the driving graph cache (retry transient download failures when building it)
G = get_street_network()
network_nodes = ox.graph_to_gdfs(G, edges = False)

In [ ]:
# Keep a smaller subgraph and its node table for the next walkshed
def get_subgraph(zone, graph, nodes):
    nodes_in_zone = nodes[nodes.geometry.within(zone)]

    return graph.subgraph(nodes_in_zone.index.to_list()), nodes_in_zone

In [ ]:
def get_lsoa_street_networks(lsoa_id, G):
    graphs = {}
    node_tables = {}
    results = {}

    for minutes, zones in extract_zones_by_duration(lsoa_id, buffer = 15).items():
        street_networks = []

        for seed, zone in zones:
            nodes = node_tables.get(seed)

            if nodes is None:
                nodes = network_nodes.iloc[network_nodes.sindex.query(zone, predicate = 'contains', sort = True)]

            graph, nodes = get_subgraph(zone, graphs.get(seed, G), nodes)
            graphs[seed] = graph
            node_tables[seed] = nodes
            street_networks.append((graph, zone))

        results[minutes] = street_networks

    return results

In [ ]:
# For a given subgraph, calculate the average block length
def calculate_avg_block_length(G):
    total_length = 0.0
    n_edges = 0

    for u, v, _, data in G.edges(keys = True, data = True):
        if isinstance(data.get('length'), (int, float)):
            segment_length = float(data['length'])
        elif 'geometry' in data and data['geometry'] is not None:
            segment_length = data['geometry'].length
        else:
            x1, y1 = G.nodes[u]['x'], G.nodes[u]['y']
            x2, y2 = G.nodes[v]['x'], G.nodes[v]['y']
            segment_length = geodesic((y1, x1), (y2, x2)).meters

        total_length += segment_length
        n_edges += 1

    return total_length / n_edges if n_edges else 0.0

In [ ]:
# Calculate the 7 street metrics for a given walkshed
def calculate_metrics(G, zone_area):
    n = G.number_of_nodes()
    e = G.number_of_edges()

    nodes = ox.graph_to_gdfs(G, edges = False)

    basic_stats = ox.basic_stats(G, area = zone_area)
    intersections = nodes[nodes['street_count'] > 2]
    dead_ends = nodes[nodes['street_count'] == 1]

    G_ig = ig.Graph.from_networkx(G)
    betweenness_vals = G_ig.betweenness(directed = False, weights = 'length')
    betweenness_avg = np.mean([x / ((n - 1) * (n - 2)) for x in betweenness_vals])

    block_length_avg = calculate_avg_block_length(G)

    return {
        'n': n,
        'e': e,
        'area': zone_area,
        'street_intersection_density': len(intersections) / zone_area,
        'street_length_density': basic_stats['street_length_total'] / zone_area,
        'street_intersection_degree_avg': np.nanmean(nodes['street_count']),
        'street_circuity_avg': basic_stats['circuity_avg'],
        'street_dead_end_density': len(dead_ends) / n,
        'street_betweenness_avg': betweenness_avg,
        'street_block_length_avg': block_length_avg
    }

In [ ]:
# Minimum number of nodes to consider a valid LSOA
NODE_CUTOFF = 60

def process_lsoa(lsoa):
    results = {}

    for minutes, subgraphs in get_lsoa_street_networks(lsoa, G).items():
        metrics = []

        for subgraph, zone in subgraphs:
            if subgraph.number_of_nodes() < NODE_CUTOFF:
                continue

            metrics.append(calculate_metrics(subgraph, zone.area / 1e6))

        if not metrics:
            results[minutes] = np.nan
            continue

        stats = pd.DataFrame(metrics).mean().to_dict()
        stats['lsoa'] = lsoa
        results[minutes] = stats

    return results

In [ ]:
# Avoid forking native geospatial libraries after loading the graph
network_nodes.sindex
results = parallelize(process_lsoa, LSOA_IDS, max_workers = 4, use_threads = True)

In [ ]:
lsoa_base = gpd.read_parquet(paths.preprocessing.shared.lsoa)
results_by_duration = {minutes: [result[minutes] for result in results] for minutes in (20, 15, 10)}

In [ ]:
component_data = {}

for minutes, duration_results in results_by_duration.items():
    valid = [result for result in duration_results if isinstance(result, dict)]
    metrics_by_lsoa = {
        result['lsoa']: {key: value for key, value in result.items() if key != 'lsoa'} for result in valid
    }
    metric_keys = list(metrics_by_lsoa[next(iter(metrics_by_lsoa))].keys())
    
    # Add the metric values to the LSOA GeoDataFrame
    lsoa = lsoa_base.copy()

    for metric in metric_keys:
        lsoa[metric] = lsoa['LSOA21CD'].map(lambda x: metrics_by_lsoa.get(x, {}).get(metric, np.nan)).astype(float)

    component_data[minutes] = lsoa

In [ ]:
def add_street_score(gdf):
    metric_direction = {
        'street_intersection_density': +1,
        'street_length_density': +1,
        'street_intersection_degree_avg': +1,
        'street_circuity_avg': -1,
        'street_dead_end_density': -1,
        'street_betweenness_avg': -1,
        'street_block_length_avg': -1
    }

    z_cols = []

    for m, sign in metric_direction.items():
        zcol = f'{m}_z'
        gdf[zcol] = (gdf[m] - gdf[m].mean()) / gdf[m].std(ddof = 0)
        gdf[zcol] *= sign

        z_cols.append(zcol)

    gdf['street_score'] = gdf[z_cols].mean(axis = 1)

    return gdf

for minutes, lsoa in component_data.items():
    component_data[minutes] = add_street_score(lsoa)

In [ ]:
# Only retain the relevant columns
columns = [
    'LSOA21CD',
    'geometry',
    'street_score',
    'street_intersection_density',
    'street_length_density',
    'street_intersection_degree_avg',
    'street_circuity_avg',
    'street_dead_end_density',
    'street_betweenness_avg',
    'street_block_length_avg'
]

for minutes, lsoa in component_data.items():
    lsoa[columns].to_parquet(paths.preprocessing.walkability[minutes].streets, index = False)

In [ ]:
def visualise_metrics():
    cm = {
        'area': 'viridis',
        'street_intersection_density': 'viridis',
        'street_length_density': 'viridis',
        'street_intersection_degree_avg': 'viridis',
        'street_circuity_avg': 'viridis_r',
        'street_dead_end_density': 'viridis_r',
        'street_betweenness_avg': 'viridis_r',
        'street_block_length_avg': 'viridis_r'
    }

    m = lsoa.explore(
        column = 'street_score',
        cmap = 'viridis',
        scheme = 'quantiles',
        k = 50,
        legend = True,
        tiles = 'CartoDB positron',
        style_kwds = {'weight': 0.5, 'lineOpacity': 0.5, 'fillOpacity': 0.4},
    )

    return m

In [ ]:
for minutes, lsoa in component_data.items():
    FILES = paths.preprocessing.walkability[minutes]
    
    m = visualise_metrics()

m